# Comparación de Qwen2.5-7B y Qwen2.5-1.5B: BASE vs LoRA

Notebook nuevo e independiente de `comparacion_finetuning_prompts.ipynb`. Evalúa siete variantes: BASE + zero/one/few LoRA de Qwen2.5-7B y BASE + zero/one/few LoRA de Qwen2.5-1.5B.

- Reutiliza las predicciones completas si ya existen; reanuda CSV parciales por índice.
- No modifica el notebook anterior ni los CSV originales: usa `results/rag_prompt_comparison_6models/`.
- Mismo split (80/20, seed 42), contextos RAG locales compartidos y métricas EM, F1, BLEU-1, ROUGE-1.
- Los modelos base usan el prompt zero; cada adaptador usa su estrategia correspondiente.
- La caché se valida también por el número/índices de ejemplos y columnas requeridas.

In [ ]:
from pathlib import Path
import sys, os, json, time, gc, random
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "src" / "preprocess" / "preprocess.py").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "preprocess" / "preprocess.py").exists():
    raise FileNotFoundError("Abre Jupyter desde la raíz de Practica_PLN_II.")
sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
MAX_LENGTH = 384
MAX_NEW_TOKENS = 32
PRINT_EVERY = 10
WARMUP_QUESTIONS = 1
LIMIT_TEST = None  # 5 para una prueba rápida; None para todo el test

MODEL_IDS = {
    "7B": "Qwen/Qwen2.5-7B-Instruct",
    "1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
}
ADAPTERS = {
    "7B_FT_zero": PROJECT_ROOT / "models" / "prompt_training_controlled" / "qwen_lora_controlled_zero",
    "7B_FT_one": PROJECT_ROOT / "models" / "prompt_training_controlled" / "qwen_lora_controlled_one",
    "7B_FT_few": PROJECT_ROOT / "models" / "prompt_training_controlled" / "qwen_lora_controlled_few",
    "1.5B_FT_zero": PROJECT_ROOT / "models" / "prompt_training_controlled_qwen15" / "qwen15_lora_controlled_zero",
    "1.5B_FT_one": PROJECT_ROOT / "models" / "prompt_training_controlled_qwen15" / "qwen15_lora_controlled_one",
    "1.5B_FT_few": PROJECT_ROOT / "models" / "prompt_training_controlled_qwen15" / "qwen15_lora_controlled_few",
}
PROMPT_FILES = {"zero": "controlled_zero.md", "one": "controlled_one.md", "few": "controlled_few.md"}
INDEX_PATH = PROJECT_ROOT / "models" / "context_embeddings.pt"
QUESTIONS_PATH = PROJECT_ROOT / "data" / "questions_train.json"
CONTEXTS_PATH = PROJECT_ROOT / "data" / "contexts.json"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
OUTPUT_DIR = PROJECT_ROOT / "results" / "rag_prompt_comparison_6models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else (
    torch.float16 if DEVICE.type == "cuda" else torch.float32
)
print("Raíz:", PROJECT_ROOT, "| dispositivo:", DEVICE, "| dtype:", DTYPE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Resultados nuevos:", OUTPUT_DIR)
for p in [QUESTIONS_PATH, CONTEXTS_PATH, INDEX_PATH]:
    print(("OK" if p.exists() else "FALTA"), p)


## 2. Datos, split y contextos RAG
Se mantiene el mismo split y se recupera el contexto una sola vez para compartirlo entre los siete modelos.

In [ ]:
from src.preprocess.preprocess import prepare_dataset
from src.evaluate import exact_match_score, f1_score, bleu_score, rouge_score
from src.retrieval.context_retriever import ContextRetriever

for path in (QUESTIONS_PATH, CONTEXTS_PATH, INDEX_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Falta archivo requerido: {path}")

dataset = prepare_dataset(str(QUESTIONS_PATH), str(CONTEXTS_PATH))
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=SEED)
train_data, test_data = [dict(x) for x in train_data], [dict(x) for x in test_data]
if LIMIT_TEST is not None:
    test_data = test_data[:LIMIT_TEST]
print(f"Total={len(dataset)} | train={len(train_data)} | test={len(test_data)}")

retriever = ContextRetriever(str(INDEX_PATH), device=DEVICE)
retrieved_test = []
for i, item in enumerate(test_data):
    r = retriever.retrieve(item["question"])
    retrieved_test.append({
        **item,
        "context": r.get("context", "") or "",
        "retrieved_context_id": r.get("context_id", "none"),
        "retrieval_score": float(r.get("score", 0.0)),
    })
    if (i + 1) % PRINT_EVERY == 0 or i + 1 == len(test_data):
        print(f"[RAG] {i+1}/{len(test_data)}", flush=True)

# Construir ejemplos de demostración únicamente a partir de train.
retrieved_train = []
for i, item in enumerate(train_data):
    r = retriever.retrieve(item["question"])
    retrieved_train.append({**item, "context": r.get("context", "") or ""})
    if (i + 1) % 50 == 0 or i + 1 == len(train_data):
        print(f"[Train demos] {i+1}/{len(train_data)}", flush=True)

answerable = [x for x in retrieved_train if x.get("has_answer") and x.get("answer") and x.get("context")]
unanswerable = [x for x in retrieved_train if not x.get("has_answer")]
if len(answerable) < 3:
    raise ValueError("No hay suficientes ejemplos respondibles para crear prompts.")
short = min(answerable, key=lambda x: len(x["answer"].split()))
medium_candidates = [x for x in answerable if x is not short and 1 <= len(x["answer"].split()) <= 5]
medium = medium_candidates[0] if medium_candidates else next(x for x in answerable if x is not short)
long_candidates = [x for x in answerable if x is not short and x is not medium]
examples = [short, medium, max(long_candidates, key=lambda x: len(x["answer"].split()))]
if unanswerable:
    examples.append(unanswerable[0])
examples = [{"context": x.get("context",""), "question": x["question"], "answer": x.get("answer") or ""} for x in examples[:4]]

def make_prompt(strategy):
    path = PROMPTS_DIR / PROMPT_FILES[strategy]
    if not path.exists():
        raise FileNotFoundError(f"No encuentro {path}. Comprueba que existen los prompts controlados.")
    template = path.read_text(encoding="utf-8").strip()
    if strategy == "zero":
        return template
    if strategy == "one":
        e = examples[0]
        return template.format(example_context=e["context"], example_question=e["question"], example_answer=e["answer"])
    values = {}
    for i, e in enumerate(examples[:4], start=1):
        values[f"example_{i}_context"] = e["context"]
        values[f"example_{i}_question"] = e["question"]
        values[f"example_{i}_answer"] = e["answer"]
    return template.format(**values)

SYSTEM_PROMPTS = {s: make_prompt(s) for s in ("zero", "one", "few")}
print("Longitudes prompts:", {k: len(v) for k,v in SYSTEM_PROMPTS.items()})


## 3. Generación, métricas y caché
Los nombres de salida son únicos para cada familia y variante. Si existe un CSV completo se reutiliza sin cargar el modelo; si está parcial, solo se generan los índices que faltan.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_tokenizer(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.truncation_side = "left"
    return tok

def load_model(model_id, adapter_path=None):
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE)
    if adapter_path is not None:
        if not (Path(adapter_path) / "adapter_config.json").exists():
            raise FileNotFoundError(f"No encuentro adapter_config.json en {adapter_path}")
        model = PeftModel.from_pretrained(model, str(adapter_path))
    return model.eval()

def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.inference_mode()
def generate_one(model, tokenizer, item, system_prompt):
    context = item.get("context") or "No context is available."
    messages = [
        {"role":"system","content":system_prompt},
        {"role":"user","content":f"Context:\n{context}\n\nQuestion: {item['question']}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k:v.to(DEVICE) for k,v in inputs.items()}
    sync_device(); t0 = time.perf_counter()
    outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    sync_device(); seconds = time.perf_counter() - t0
    ids = outputs[:, inputs["input_ids"].shape[1]:]
    prediction = tokenizer.batch_decode(ids, skip_special_tokens=True)[0].strip()
    return prediction, seconds, int(ids.shape[1]), int(inputs["input_ids"].shape[1])

def calc_record(variant, family, strategy, i, item, prediction, gen_seconds, n_tokens, input_tokens, error=None):
    reference = item.get("answer") or ""
    return {
        "variant":variant, "family":family, "prompt_strategy":strategy, "index":i,
        "question":item["question"], "reference":reference, "prediction":prediction,
        "has_answer":bool(item["has_answer"]), "retrieved_context_id":item["retrieved_context_id"],
        "retrieval_score":item["retrieval_score"], "retrieved_context":item["context"],
        "generation_seconds":gen_seconds, "generated_tokens":n_tokens, "input_tokens":input_tokens,
        "tokens_per_second":n_tokens/gen_seconds if gen_seconds and gen_seconds > 0 else np.nan,
        "em":float(exact_match_score(prediction, reference)), "f1":float(f1_score(prediction, reference)),
        "BLEU_percent":100*bleu_score(prediction, reference), "ROUGE_percent":100*rouge_score(prediction, reference),
        "empty_prediction":not bool(prediction.strip()), "error":error,
    }

def summarize(records):
    good = [r for r in records if not r.get("error")]
    ans = [r for r in good if bool(r.get("has_answer"))]
    unans = [r for r in good if not bool(r.get("has_answer"))]
    def avg(rows, key):
        vals = pd.to_numeric(pd.Series([r.get(key, np.nan) for r in rows]), errors="coerce").dropna()
        return float(vals.mean()) if len(vals) else np.nan
    lat = [r.get("generation_seconds", np.nan) for r in good]
    lat = [float(x) for x in lat if pd.notna(x)]
    return {
        "n_total":len(records), "n_success":len(good), "n_errors":sum(bool(r.get("error")) for r in records),
        "EM_percent":100*avg(good,"em"), "F1_percent":100*avg(good,"f1"),
        "answerable_F1_percent":100*avg(ans,"f1"), "unanswerable_F1_percent":100*avg(unans,"f1"),
        "BLEU_percent":avg(good,"BLEU_percent"), "ROUGE_percent":avg(good,"ROUGE_percent"),
        "empty_prediction_rate_percent":100*avg(good,"empty_prediction"),
        "generation_mean_seconds":float(np.mean(lat)) if lat else np.nan,
        "generation_p95_seconds":float(np.percentile(lat,95)) if lat else np.nan,
        "mean_tokens_per_second":avg(good,"tokens_per_second"),
    }

VARIANTS = [
    {"name":"7B_BASE","family":"7B","model_id":MODEL_IDS["7B"],"adapter":None,"strategy":"zero"},
    {"name":"7B_FT_zero","family":"7B","model_id":MODEL_IDS["7B"],"adapter":ADAPTERS["7B_FT_zero"],"strategy":"zero"},
    {"name":"7B_FT_one","family":"7B","model_id":MODEL_IDS["7B"],"adapter":ADAPTERS["7B_FT_one"],"strategy":"one"},
    {"name":"7B_FT_few","family":"7B","model_id":MODEL_IDS["7B"],"adapter":ADAPTERS["7B_FT_few"],"strategy":"few"},
    {"name":"1.5B_BASE","family":"1.5B","model_id":MODEL_IDS["1.5B"],"adapter":None,"strategy":"zero"},
    {"name":"1.5B_FT_zero","family":"1.5B","model_id":MODEL_IDS["1.5B"],"adapter":ADAPTERS["1.5B_FT_zero"],"strategy":"zero"},
    {"name":"1.5B_FT_one","family":"1.5B","model_id":MODEL_IDS["1.5B"],"adapter":ADAPTERS["1.5B_FT_one"],"strategy":"one"},
    {"name":"1.5B_FT_few","family":"1.5B","model_id":MODEL_IDS["1.5B"],"adapter":ADAPTERS["1.5B_FT_few"],"strategy":"few"},
]
expected_n = len(retrieved_test)
required_cols = {"index","prediction","reference","question","error"}

def read_cache(name):
    final_path = OUTPUT_DIR / f"{name.lower()}_predictions.csv"
    partial_path = OUTPUT_DIR / f"{name.lower()}_predictions_partial.csv"
    for path in (final_path, partial_path):
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
            if not required_cols.issubset(df.columns):
                print(f"[CACHÉ INVÁLIDA] {path.name}: faltan columnas requeridas.")
                continue
            df["index"] = pd.to_numeric(df["index"], errors="coerce")
            df = df.dropna(subset=["index"])
            df["index"] = df["index"].astype(int).drop_duplicates()
            df = df.drop_duplicates(subset=["index"], keep="last").sort_values("index")
            complete = (len(df)==expected_n and df["index"].tolist()==list(range(expected_n))
                        and df["error"].isna().all())
            print(f"[CACHÉ] {name}: {len(df)}/{expected_n} ({'completa' if complete else 'parcial'})")
            return df, complete
        except Exception as exc:
            print(f"No se pudo leer {path.name}: {exc}")
    return pd.DataFrame(), False

print("Variantes configuradas:", [v["name"] for v in VARIANTS])


## 4. Ejecutar los modelos pendientes
El modelo se carga únicamente cuando su caché no está completa. Los errores individuales quedan registrados y pueden reintentarse al volver a ejecutar el notebook.

In [ ]:
set_seed()
summaries = []
all_records = []

for cfg in VARIANTS:
    name, family, strategy = cfg["name"], cfg["family"], cfg["strategy"]
    print("\n" + "="*80 + f"\n{name} | prompt={strategy}\n" + "="*80, flush=True)
    cached, complete = read_cache(name)
    if complete:
        records = cached.to_dict("records")
        # Normalizar los nulos de pandas para métricas consistentes.
        for r in records:
            r["error"] = None if pd.isna(r.get("error")) else r.get("error")
            r["has_answer"] = bool(r.get("has_answer", False))
        s = summarize(records)
        s.update({"variant":name,"family":family,"prompt_strategy":strategy,"adapter_path":str(cfg["adapter"]) if cfg["adapter"] else "BASE"})
        summaries.append(s); all_records.extend(records)
        print("Inferencia omitida: CSV completo.", flush=True)
        continue

    adapter = cfg["adapter"]
    if adapter is not None and not Path(adapter).exists():
        print(f"OMITIDO: no existe el adaptador {adapter}. Entrena ese adaptador o corrige la ruta.")
        summaries.append({"variant":name,"family":family,"prompt_strategy":strategy,"load_error":f"Missing adapter: {adapter}"})
        continue

    by_index = {}
    for row in cached.to_dict("records"):
        idx = row.get("index")
        if pd.notna(idx) and not row.get("error"):
            by_index[int(idx)] = row
    missing = [(i,item) for i,item in enumerate(retrieved_test) if i not in by_index]
    print(f"Caché válida: {len(by_index)}; preguntas pendientes: {len(missing)}")
    if not missing:
        records = [by_index[i] for i in sorted(by_index)]
        s = summarize(records); s.update({"variant":name,"family":family,"prompt_strategy":strategy})
        summaries.append(s); all_records.extend(records); continue

    tokenizer = model = None
    try:
        tokenizer = load_tokenizer(cfg["model_id"])
        t_load = time.perf_counter()
        model = load_model(cfg["model_id"], adapter)
        load_seconds = time.perf_counter() - t_load
        print(f"Modelo cargado en {load_seconds:.1f}s")
        if WARMUP_QUESTIONS:
            _ = generate_one(model, tokenizer, retrieved_test[0], SYSTEM_PROMPTS[strategy])
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        started = time.perf_counter()
        for i,item in missing:
            try:
                pred, sec, ntok, itok = generate_one(model, tokenizer, item, SYSTEM_PROMPTS[strategy])
                rec = calc_record(name,family,strategy,i,item,pred,sec,ntok,itok)
            except Exception as exc:
                rec = calc_record(name,family,strategy,i,item,"",np.nan,0,0,error=repr(exc))
                print(f"[ERROR] {name}, pregunta {i}: {exc}", flush=True)
            by_index[i] = rec
            current = [by_index[k] for k in sorted(by_index)]
            pd.DataFrame(current).to_csv(OUTPUT_DIR / f"{name.lower()}_predictions_partial.csv", index=False, encoding="utf-8-sig")
            if len(by_index) % PRINT_EVERY == 0 or len(by_index) == expected_n:
                sm = summarize(current)
                print(f"[{name}] {len(by_index)}/{expected_n} | EM={sm['EM_percent']:.2f}% | F1={sm['F1_percent']:.2f}%", flush=True)

        records = [by_index[i] for i in sorted(by_index)]
        s = summarize(records)
        s.update({
            "variant":name,"family":family,"prompt_strategy":strategy,
            "adapter_path":str(adapter) if adapter else "BASE",
            "model_load_seconds_secondary":load_seconds,
            "peak_vram_gb":torch.cuda.max_memory_allocated()/(1024**3) if torch.cuda.is_available() else np.nan,
            "variant_wall_seconds":time.perf_counter()-started,
        })
        pd.DataFrame(records).to_csv(OUTPUT_DIR / f"{name.lower()}_predictions.csv", index=False, encoding="utf-8-sig")
        (OUTPUT_DIR / f"{name.lower()}_predictions_partial.csv").unlink(missing_ok=True)
        summaries.append(s); all_records.extend(records)
    except Exception as exc:
        print(f"[ERROR DE CARGA/EJECUCIÓN] {name}: {exc}", flush=True)
        summaries.append({"variant":name,"family":family,"prompt_strategy":strategy,"load_error":repr(exc)})
    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pd.DataFrame(summaries).to_csv(OUTPUT_DIR / "summary_metrics_partial.csv", index=False)
    pd.DataFrame(all_records).to_csv(OUTPUT_DIR / "all_predictions_partial.csv", index=False)

summary_df = pd.DataFrame(summaries)
predictions_df = pd.DataFrame(all_records)
display(summary_df)


## 5. Consolidación y comparación
Se guardan métricas y predicciones conjuntas, además de una tabla compacta y una gráfica calidad/velocidad.

In [ ]:
# Recargar también las cachés de variantes que se hayan omitido por un error de ruta/carga,
# de forma que el consolidado incluya todos los CSV individuales disponibles.
records_by_name = {}
for cfg in VARIANTS:
    name = cfg["name"]
    df, complete = read_cache(name)
    if not df.empty:
        records_by_name[name] = df

if records_by_name:
    predictions_df = pd.concat(
        [df.assign(variant=name) for name,df in records_by_name.items()],
        ignore_index=True, sort=False
    )
    predictions_df = predictions_df.drop_duplicates(subset=["variant","index"], keep="last")
    predictions_df = predictions_df.sort_values(["family","variant","index"]).reset_index(drop=True)

    # Métricas calculadas desde las predicciones almacenadas, sin inferencia adicional.
    final_summaries = []
    for cfg in VARIANTS:
        name = cfg["name"]
        if name not in records_by_name:
            continue
        recs = records_by_name[name].to_dict("records")
        for r in recs:
            r["error"] = None if pd.isna(r.get("error")) else r.get("error")
            if "has_answer" in r:
                r["has_answer"] = False if pd.isna(r["has_answer"]) else bool(r["has_answer"])
        sm = summarize(recs)
        sm.update({"variant":name,"family":cfg["family"],"prompt_strategy":cfg["strategy"],
                   "adapter_path":str(cfg["adapter"]) if cfg["adapter"] else "BASE"})
        final_summaries.append(sm)
    summary_df = pd.DataFrame(final_summaries)
    summary_df.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False, encoding="utf-8-sig")
    predictions_df.to_csv(OUTPUT_DIR / "all_predictions.csv", index=False, encoding="utf-8-sig")
    print("Guardados:", OUTPUT_DIR / "summary_metrics.csv", "y", OUTPUT_DIR / "all_predictions.csv")
    display(summary_df.sort_values(["family","F1_percent"], ascending=[True,False]).round(2))
else:
    print("Todavía no hay CSV de predicciones disponibles para consolidar.")

